In [3]:
import tensorly as tl
tl.tenalg.set_backend('einsum')
tl.plugins.use_opt_einsum()

In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
from moabb.paradigms import P300,LeftRightImagery, MotorImagery
from moabb.datasets import *
from mne.decoding import Scaler


#paradigm = LeftRightImagery(resample=250)
#dataset = BNCI2014004()

paradigm=P300(resample=48)
dataset=BNCI2014008()
epochs, labels, meta = paradigm.get_data(
    dataset=dataset, 
     subjects=[2],
     return_epochs=True
)
session = meta['session'].unique()[0]
idc = meta['session'] == session
epochs = epochs[idc]
labels = labels[idc]
meta = meta[idc]


BNCI2014008 has been renamed to BNCI2014_008. BNCI2014008 will be removed in version 1.1.
The dataset class name 'BNCI2014008' must be an abbreviation of its code 'BNCI2014-008'. See moabb.datasets.base.is_abbrev for more information.
/usr/local/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 4200 events (all good), 0 – 1 s (baseline off), ~65.9 MB, data loaded,
 'Target': 700
 'NonTarget': 3500>
  warn(f"warnEpochs {epochs}")


Adding metadata with 3 columns
Adding metadata with 3 columns
4200 matching events found
No baseline correction applied


/usr/local/lib/python3.11/site-packages/moabb/paradigms/base.py:350: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  X = mne.concatenate_epochs(X)


In [8]:
import tensorly.decomposition
import matplotlib.pyplot as plt
import tensorly as tl
from hoda.tensorize import stf_tensor, hankel_tensor
import numpy as np
from mne.decoding import Scaler

X = epochs.get_data()
X = Scaler(scalings='mean', with_mean=True).fit_transform(X)
#X = stf_tensor(X, sfreq=epochs.info['sfreq'], normalize=True, log=True, n_freqs=8, bin_freq=8)
X = hankel_tensor(X)
X = tl.tensor(X)
y = labels
X.shape

(4200, 8, 25, 24)

In [10]:
import seaborn as sns
"""
x = X.flatten()
x = tl.to_numpy(x)
sns.distplot(x)
"""

'\nx = X.flatten()\nx = tl.to_numpy(x)\nsns.distplot(x)\n'

In [11]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=1/5, random_state=2, stratify=y)

In [ ]:
from hoda.hoda import BTTDA, HODA, trunc_eigh
from hoda.cov import mode_scatter


bttda = BTTDA(
    ranks=[2]*10,
    hoda_params=dict(
        rank=None,
        max_iter=128,
        tol=1e-8,
        init ='random',
        shrinkage='lw',
        toeplitz=None,
        obj='tr',
        solver='lanczos',
        taper=False,
        extra_train_info=False,
        verbose=True,
        random_state=42,
        delta=None,
       
    ),
    verbose=True,
    extra_train_info=True,
)
#%env PYTHONWARNINGS=ignore
%time bttda.fit(X_train,y_train)

Fitting block 1/10...


Forward model :   2%|▏         | 2/128 [00:00<00:29,  4.31it/s]


Fitting block 2/10...


Forward model :   2%|▏         | 2/128 [00:00<00:37,  3.40it/s]


Fitting block 3/10...


Backward HODA model rank=(2, 2, 2):  21%|██        | 27/128 [00:07<00:26,  3.75it/s]

In [ ]:
import pandas as pd

df = pd.DataFrame(bttda.train_info_)
df

In [ ]:
import seaborn as sns
sns.lineplot(data=df, x='block', y='nmse')

In [ ]:
from hoda.hoda import forward_stats
info = []
for b in range(len(bttda.blocks_)):
    Xt_test = bttda.transform(X_test, blocks=bttda.blocks_[:b+1])
    X_rec_test = bttda.inv_transform(Xt_test, n_blocks=b+1)
    res = forward_stats(X_test, Xt_test, X_rec_test,y)
    res['block'] =b
    info.append(res)
info = pd.DataFrame(info)
sns.lineplot(data=info, x='block', y='nmse')
info

In [ ]:
from hoda.hoda import forward_stats
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score

score_train = []
score_test = []
for b in range(len(bttda.blocks_)):
    Xt_train = tl.to_numpy(bttda.transform(X_train, blocks=bttda.blocks_[:b+1]))
    Xt_test = tl.to_numpy(bttda.transform(X_test, blocks=bttda.blocks_[:b+1]))
    clf = LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
    clf.fit(Xt_train,y_train)
    score_train.append(accuracy_score(y_train, clf.predict(Xt_train)))
    score_test.append(accuracy_score(y_test, clf.predict(Xt_test)))
plt.plot(score_train)
plt.plot(score_test)

In [ ]:
Xt = bttda.transform(X)
Xt = tl.to_numpy(Xt)
corr = np.corrcoef(Xt, rowvar=False)
sns.heatmap(corr, center=0, cmap='vlag')